# 02 Cleaning and analysis-table preparation

This notebook validates the two processed source tables and
creates a 24-row parish-level analysis table. A pilot commercial
asking-rent sample is available for three priority parishes;
the citywide output therefore remains provisional.

## 1. Imports and project configuration

Project paths and reusable validation/scoring functions are
imported from `src/`. All output paths are repository-relative.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    COMMERCIAL_RENT_PARISH_FILE,
    COWORKING_LOCATIONS_FILE,
    PARISH_ANALYSIS_BASE_FILE,
    PARISH_INDICATORS_FILE,
    TABLES_DIR,
)
from src.utils import inverse_percentile_score, validate_required_columns

TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and validate processed inputs

The processed location inventory is retained as an audit table.
The parish indicators table is the input for the decision
analysis and must contain exactly one row per Lisbon parish.

In [2]:
coworking = pd.read_csv(
    COWORKING_LOCATIONS_FILE,
    parse_dates=["collection_date"],
)
parish = pd.read_csv(
    PARISH_INDICATORS_FILE,
    parse_dates=["collection_date"],
)
rent = pd.read_csv(
    COMMERCIAL_RENT_PARISH_FILE,
    parse_dates=["collection_date"],
)

required_coworking = {
    "coworking_id",
    "parish",
    "active_status",
    "verification_status",
}
required_parish = {
    "parish_code",
    "parish",
    "area_km2",
    "working_age_population_15_64",
    "coworking_count",
    "coworking_per_10000_working_age",
    "demand_proxy_score",
    "transit_access_score",
    "median_rent_eur_m2_month",
    "rent_sample_size",
}
validate_required_columns(coworking, required_coworking)
validate_required_columns(parish, required_parish)

print("Coworking audit table:", coworking.shape)
print("Parish indicator table:", parish.shape)
print("Rent pilot summary:", rent.shape)
print(
    "Verified active locations:",
    int(
        (
            coworking["verification_status"]
            == "verified_official_site"
        ).sum()
    ),
)
parish.head()

Coworking audit table: (54, 21)
Parish indicator table: (24, 39)
Rent pilot summary: (7, 8)
Verified active locations: 48


,parish_code,parish,area_km2,population_total,working_age_population_15_64,coworking_count,pending_coworking_count,coworking_per_10000_working_age,office_count,cafe_count,...,transit_access_score,demand_proxy_score,median_rent_eur_m2_month,rent_sample_size,rent_coverage_flag,opportunity_score,poi_query_status,population_reference_year,collection_date,coworking_status_note
0,110601,Ajuda,2.8739,14306,8591,0,0,0.000,10,41,...,33.75,37.92,NaN,0,not_collected,NaN,success,2021,2026-07-25,Count includes verified active rows with assig...
1,110602,Alcântara,5.0697,13850,8735,2,0,2.290,31,40,...,12.08,32.08,NaN,0,not_collected,NaN,success,2021,2026-07-25,Count includes verified active rows with assig...
2,110654,Alvalade,5.3366,33309,20433,4,0,1.958,63,58,...,65.83,60.00,NaN,0,not_collected,NaN,success,2021,2026-07-25,Count includes verified active rows with assig...
3,110655,Areeiro,1.7173,21160,13319,0,0,0.000,45,47,...,73.33,85.42,NaN,0,not_collected,NaN,success,2021,2026-07-25,Count includes verified active rows with assig...
4,110656,Arroios,2.1259,33302,23171,7,0,3.021,55,106,...,95.00,94.17,NaN,0,not_collected,NaN,success,2021,2026-07-25,Count includes verified active rows with assig...


## 3. Select analysis fields and derive competition opportunity

Lower coworking supply per 10,000 working-age residents receives
a higher competition-opportunity score. This is a relative
screening measure, not proof of unmet demand.

In [3]:
analysis_columns = [
    "parish_code",
    "parish",
    "area_km2",
    "population_total",
    "working_age_population_15_64",
    "working_age_population_density_km2",
    "coworking_count",
    "coworking_per_10000_working_age",
    "coworking_density_km2",
    "office_count",
    "office_density_km2",
    "cafe_count",
    "cafe_density_km2",
    "hotel_count",
    "hotel_density_km2",
    "municipal_higher_education_count",
    "municipal_higher_education_density_km2",
    "municipal_metro_station_count",
    "municipal_metro_station_density_km2",
    "bus_tram_stop_count",
    "bus_tram_stop_density_km2",
    "demand_proxy_score",
    "transit_access_score",
    "median_rent_eur_m2_month",
    "rent_sample_size",
    "rent_coverage_flag",
    "population_reference_year",
    "collection_date",
]

analysis = parish[analysis_columns].copy()
analysis["competition_opportunity_score"] = inverse_percentile_score(
    analysis["coworking_per_10000_working_age"]
)
usable_rent = rent[
    rent["rent_coverage_flag"].isin(
        ["target_met", "usable_low_coverage"]
    )
][
    [
        "parish",
        "median_rent_eur_m2_month",
        "rent_sample_size",
        "rent_coverage_flag",
    ]
]
analysis = analysis.drop(
    columns=[
        "median_rent_eur_m2_month",
        "rent_sample_size",
        "rent_coverage_flag",
    ]
).merge(
    usable_rent,
    on="parish",
    how="left",
    validate="one_to_one",
)
analysis["rent_sample_size"] = (
    analysis["rent_sample_size"].fillna(0).astype(int)
)
analysis["rent_coverage_flag"] = analysis[
    "rent_coverage_flag"
].fillna("not_collected")
analysis["analysis_status"] = "provisional_partial_rent"
analysis = analysis.sort_values("parish").reset_index(drop=True)
analysis.head()

,parish_code,parish,area_km2,population_total,working_age_population_15_64,working_age_population_density_km2,coworking_count,coworking_per_10000_working_age,coworking_density_km2,office_count,...,bus_tram_stop_density_km2,demand_proxy_score,transit_access_score,population_reference_year,collection_date,competition_opportunity_score,median_rent_eur_m2_month,rent_sample_size,rent_coverage_flag,analysis_status
0,110601,Ajuda,2.8739,14306,8591,2989.318,0,0.000,0.000,10,...,29.577,37.92,33.75,2021,2026-07-25,100.00,NaN,0,not_collected,provisional_partial_rent
1,110602,Alcântara,5.0697,13850,8735,1722.982,2,2.290,0.395,31,...,16.175,32.08,12.08,2021,2026-07-25,21.62,NaN,0,not_collected,provisional_partial_rent
2,110654,Alvalade,5.3366,33309,20433,3828.842,4,1.958,0.750,63,...,28.670,60.00,65.83,2021,2026-07-25,37.84,NaN,0,not_collected,provisional_partial_rent
3,110655,Areeiro,1.7173,21160,13319,7755.779,0,0.000,0.000,45,...,30.862,85.42,73.33,2021,2026-07-25,100.00,17.73,10,target_met,provisional_partial_rent
4,110656,Arroios,2.1259,33302,23171,10899.384,7,3.021,3.293,55,...,47.509,94.17,95.00,2021,2026-07-25,10.81,17.50,15,target_met,provisional_partial_rent


## 4. Quality checks

The base table must contain all 24 parishes, reconcile to 48
verified active locations and have complete core analysis
fields. The pilot rent medians must be available only where at
least five distinct building observations support them.

In [4]:
core_columns = [
    "parish_code",
    "parish",
    "working_age_population_15_64",
    "coworking_count",
    "demand_proxy_score",
    "transit_access_score",
    "competition_opportunity_score",
]

assert len(analysis) == 24
assert analysis["parish"].nunique() == 24
assert analysis["parish_code"].nunique() == 24
assert int(analysis["coworking_count"].sum()) == 48
assert not analysis[core_columns].isna().any().any()
assert analysis[
    [
        "demand_proxy_score",
        "transit_access_score",
        "competition_opportunity_score",
    ]
].apply(lambda column: column.between(0, 100).all()).all()
assert analysis["median_rent_eur_m2_month"].notna().sum() == 3
assert analysis.loc[
    analysis["median_rent_eur_m2_month"].notna(),
    "rent_sample_size",
].ge(5).all()

quality_summary = pd.DataFrame(
    {
        "check": [
            "row_count",
            "unique_parishes",
            "verified_coworking_total",
            "core_missing_values",
            "rent_values_available",
        ],
        "value": [
            len(analysis),
            analysis["parish"].nunique(),
            int(analysis["coworking_count"].sum()),
            int(analysis[core_columns].isna().sum().sum()),
            int(analysis["median_rent_eur_m2_month"].notna().sum()),
        ],
    }
)
quality_summary

,check,value
0,row_count,24
1,unique_parishes,24
2,verified_coworking_total,48
3,core_missing_values,0
4,rent_values_available,3


## 5. Export the base analysis table

The CSV is the common source for EDA, provisional scoring and
the future BI dashboard. A separate quality summary makes the
checkpoint easy to audit.

In [5]:
quality_path = TABLES_DIR / "parish_analysis_quality_summary.csv"
analysis.to_csv(PARISH_ANALYSIS_BASE_FILE, index=False)
quality_summary.to_csv(quality_path, index=False)

print(
    "Saved analysis table:",
    PARISH_ANALYSIS_BASE_FILE.relative_to(PROJECT_ROOT),
)
print("Saved quality summary:", quality_path.relative_to(PROJECT_ROOT))
print("Analysis table shape:", analysis.shape)

Saved analysis table: data/processed/parish_analysis_base.csv
Saved quality summary: reports/tables/parish_analysis_quality_summary.csv
Analysis table shape: (24, 30)


## 6. Cleaning outcome

The analysis base contains 24 unique parishes and reconciles to
48 verified active coworking locations. Demand, accessibility
and competition components are complete. Rent is usable for
Areeiro, Arroios and Campolide only; missing rent elsewhere is
not treated as zero.